# Milestone 2 lab, notebook 4 of 8: Land it raw

## Where we are

| Notebook | What you do |
|---|---|
| 1. Ask only for what changed | Read the orders that changed, not all of them |
| 2. Remember where you stopped | Keep a watermark. See what happens when it moves too early |
| 3. When the API says no | Meet the rate limit and a server error. Wait, and try again |
| **4. Land it raw** (you are here) | Put every order in Postgres, exactly as it arrived |
| 5. Pull or push | A short lab: let the order system send the changes to you |
| 6. What you built has a name | Change data capture, and what this kind cannot see |
| 7. Counted twice | Run a few mornings. Watch the report go wrong |
| 8. The production version | The same logic as a tested program |

In milestone 1 the API's answer was used once and thrown away. When someone asked a question about products, the warehouse could not answer.

From now on, **everything that arrives is kept, exactly as it arrived**. And the warehouse becomes Postgres.

In [1]:
from psycopg.types.json import Jsonb

from lab_helpers import lab_warehouse, read_orders, set_morning, show_doc, table

con = lab_warehouse(fresh=True)       # the lab's own empty database
table(con, "SELECT current_database() AS database, split_part(version(), ' on ', 1) AS warehouse")

,database,warehouse
0,arden_m2_lab,PostgreSQL 16.13 (Debian 16.13-1.pgdg13+1)


`lab_warehouse` connects to the Postgres that Docker runs. If that cell failed, start it from the project folder with `docker compose up -d warehouse`.

## Step 1: A table for raw data

Make it by hand. It has one column for the order, and three columns about the **load**.

In [2]:
con.execute("""
    CREATE SCHEMA raw;

    CREATE TABLE raw.orders_landing (
        landing_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        payload     JSONB NOT NULL,          -- the order, exactly as received
        run_id      TEXT NOT NULL,           -- which run loaded it
        page_no     INTEGER NOT NULL,        -- on which page it arrived
        loaded_at   TIMESTAMPTZ NOT NULL     -- when it was loaded
    );
""")
print("Table created.")

Table created.


`JSONB` is Postgres's type for JSON. The whole order goes into one cell, with its items and every field. If the order system adds a field tomorrow, this table still takes it.

## Step 2: Land Monday

One row per order. Nothing is cleaned, renamed or dropped.

In [3]:
def land(orders, run_id):
    """Add the orders at the end of the landing table."""
    with con.cursor() as cursor:
        cursor.executemany(
            "INSERT INTO raw.orders_landing (payload, run_id, page_no, loaded_at) VALUES (%s, %s, %s, now())",
            [(Jsonb(order), run_id, 1 + i // 25) for i, order in enumerate(orders)],
        )

set_morning("2026-06-08")
monday, calls = read_orders()
land(monday, run_id="2026-06-08_1")

table(con, "SELECT count(*) AS rows_landed FROM raw.orders_landing")

It is now Monday 2026-06-08 07:00:00 in Arden.


,rows_landed
0,226


In [4]:
table(con, "SELECT landing_id, run_id, page_no, loaded_at, payload FROM raw.orders_landing ORDER BY landing_id LIMIT 3")

,landing_id,run_id,page_no,loaded_at,payload
0,1,2026-06-08_1,1,2026-11-21 09:32:08.969309+00:00,"{'city': 'NGT', 'items': [{'line_no': 1, 'quan..."
1,2,2026-06-08_1,1,2026-11-21 09:32:08.969309+00:00,"{'city': 'LKS', 'items': [{'line_no': 1, 'quan..."
2,3,2026-06-08_1,1,2026-11-21 09:32:08.969309+00:00,"{'city': 'RVB', 'items': [{'line_no': 1, 'quan..."


## Step 3: Read it like a table

SQL can reach inside the JSON. `payload->>'city'` means "the `city` field, as text".

In [5]:
table(con, """
    SELECT (payload->>'order_id')::int                                               AS order_id,
           payload->>'city'                                                          AS city,
           payload->>'status'                                                        AS status,
           ((payload->>'placed_at')::timestamptz AT TIME ZONE 'America/Chicago')::date AS order_date,
           (payload->>'total_cents')::int                                            AS total_cents
    FROM raw.orders_landing
    WHERE (payload->>'order_id')::int IN (1130, 1196, 1197)
""")

,order_id,city,status,order_date,total_cents
0,1130,NGT,delivered,2026-06-05,1475
1,1196,RVB,delivered,2026-06-06,1796
2,1197,LKS,delivered,2026-06-07,1904


Order 1196, the two-minutes-to-midnight order, is on Saturday 6 June. The time zone rule from milestone 1 is now one line of SQL.

## Step 4: The question milestone 1 could not answer

*Which product sold the most units on Sunday?* In milestone 1 the items were dropped before the load. Now they are here.

In [6]:
table(con, """
    SELECT item->>'product_name'            AS product,
           sum((item->>'quantity')::int)    AS units_sold
    FROM raw.orders_landing,
         jsonb_array_elements(payload->'items') AS item
    WHERE ((payload->>'placed_at')::timestamptz AT TIME ZONE 'America/Chicago')::date = '2026-06-07'
      AND payload->>'status' <> 'cancelled'
      AND item->>'item_status' = 'ok'
    GROUP BY 1
    ORDER BY units_sold DESC, product
    LIMIT 5
""")

,product,units_sold
0,Basmati rice 1kg,12
1,Butter 250g,12
2,Chicken breast 500g,11
3,Baby spinach,10
4,Mozzarella,9


Nobody called the order system again. The answer was already in the warehouse.

> Save the raw rows first, exactly as they arrived. Then clean in a second step. You can redo the second step without calling the source again.

## Step 5: Tuesday is added, not swapped in

The watermark now comes from the table itself: the largest `updated_at` it holds.

In [7]:
watermark = table(con, "SELECT max(payload->>'updated_at') AS watermark FROM raw.orders_landing")["watermark"][0]
print("Watermark:", watermark)

set_morning("2026-06-09")
tuesday, calls = read_orders(updated_since=watermark)
land(tuesday, run_id="2026-06-09_1")

table(con, """
    SELECT run_id, count(*) AS rows_landed, min(loaded_at) AS loaded_at
    FROM raw.orders_landing GROUP BY run_id ORDER BY run_id
""")

Watermark: 2026-06-08T03:48:53Z
It is now Tuesday 2026-06-09 07:00:00 in Arden.


,run_id,rows_landed,loaded_at
0,2026-06-08_1,226,2026-11-21 09:32:08.969309+00:00
1,2026-06-09_1,29,2026-11-21 09:32:08.995196+00:00


Monday's 226 rows were not touched. Tuesday's 29 were added at the end. That way of writing is called **append**.

The three extra columns are **load metadata**: facts about the load, not about the order. They answer questions like this one: *what did we know about order 1130, and when?*

In [8]:
table(con, """
    SELECT landing_id, run_id,
           payload->>'status'                   AS status,
           (payload->>'total_cents')::int       AS total_cents,
           (payload->>'refunded_cents')::int    AS refunded_cents,
           payload->>'updated_at'               AS updated_at
    FROM raw.orders_landing
    WHERE (payload->>'order_id')::int = 1130
    ORDER BY landing_id
""")

,landing_id,run_id,status,total_cents,refunded_cents,updated_at
0,137,2026-06-08_1,delivered,1475,0,2026-06-05T20:45:00Z
1,232,2026-06-09_1,delivered,1475,578,2026-06-08T16:00:00Z


Two rows for one order. Monday's run saw it before the refund. Tuesday's run saw it after. Finance can now explain exactly why Friday's number moved.

Notice it, though: **order 1130 is in this table twice.** Notebook 7 is about that.

## Step 6: The watermark moves in with the rows

In notebook 2 the watermark lived in a file, and a crash between "write the rows" and "save the watermark" was dangerous. A database can do both in **one transaction**: all the steps succeed, or none does.

In [9]:
con.execute("""
    CREATE SCHEMA ops;
    CREATE TABLE ops.watermarks (source TEXT PRIMARY KEY, watermark TIMESTAMPTZ NOT NULL);
    INSERT INTO ops.watermarks
    SELECT 'orders', max((payload->>'updated_at')::timestamptz) FROM raw.orders_landing;
""")
table(con, "SELECT * FROM ops.watermarks")

,source,watermark
0,orders,2026-06-09 04:17:31+00:00


In [10]:
set_morning("2026-06-10")
wednesday, _ = read_orders(updated_since="2026-06-09T04:17:31Z")

try:
    with con.transaction():                                  # everything inside is one unit
        land(wednesday, run_id="2026-06-10_1")
        raise RuntimeError("💥 the job crashes after the rows, before the watermark")
        con.execute("UPDATE ops.watermarks SET watermark = %s", [max(o["updated_at"] for o in wednesday)])
except RuntimeError as error:
    print(error)

table(con, """
    SELECT (SELECT count(*) FROM raw.orders_landing WHERE run_id = '2026-06-10_1') AS wednesday_rows_saved,
           (SELECT watermark FROM ops.watermarks)                                 AS watermark
""")

It is now Wednesday 2026-06-10 07:00:00 in Arden.
💥 the job crashes after the rows, before the watermark


,wednesday_rows_saved,watermark
0,0,2026-06-09 04:17:31+00:00


The rows were written, then the crash came, and the database took the rows back. Zero rows, and the old watermark. The next run starts from the same place.

In [11]:
show_doc("LLD.md", "5. Load (`load.py`)")

**From `docs/LLD.md`**

> ## 5. Load (`load.py`)
> 
> `land(...)` does three things in **one transaction**:
> 
> ```sql
> INSERT INTO raw.orders_landing (payload, run_id, page_no, loaded_at) VALUES ...;   -- append
> 
> INSERT INTO ops.watermarks (source, watermark, saved_by_run)
> VALUES ('orders', <largest updated_at read>, <run_id>)
> ON CONFLICT (source) DO UPDATE
>     SET watermark = greatest(ops.watermarks.watermark, EXCLUDED.watermark),        -- never moves back
>         saved_by_run = EXCLUDED.saved_by_run;
> 
> INSERT INTO ops.load_runs VALUES (...);                                             -- the log
> ```
> 
> | Rule | How |
> |---|---|
> | The watermark is the largest `updated_at` read | Not `now()`. Not the time of the run |
> | A run that reads nothing keeps the watermark | The first `INSERT` adds nothing, the second is skipped |
> | The watermark never moves back | `greatest(old, new)` |
> | A run id is readable | `2026-06-09_2` is the second run of that morning |

In [12]:
con.close()

## New words

| Word | Meaning | Here |
|---|---|---|
| **Raw landing table** | "The landing table is the raw copy of what arrived." | `raw.orders_landing`, one JSON order per row |
| **Load metadata** | Columns that describe the load, not the data: which run, when | `run_id`, `page_no`, `loaded_at` |
| **Append** | "Append adds rows at the end of a table. It never touches the rows that are already there." | Tuesday's 29 rows after Monday's 226 |

The definitions are from the D8Loop chapters [Extraction: Pulling Data Reliably](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/extraction-pulling-data-reliably) and [Loading & Write Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/loading-and-write-patterns).

## What you now know

- The raw landing table keeps every order exactly as received. A new question no longer needs a new call to the source.
- Load metadata says which run loaded a row, and when.
- Append only adds rows. Old versions of an order stay beside new ones.
- Rows and watermark are saved in one transaction: both, or neither.

**Say it in an interview:** "I land the API's response as JSON in an append-only raw table, with the run id and load time on every row. When a new question came up about products, I answered it from the raw table without touching the source. The watermark is updated in the same transaction as the insert."

Next: [notebook 5](05_pull_or_push.ipynb) is a short lab. What if the order system sent you the changes, and you did not have to ask?